In [ ]:
%load_ext autoreload
%autoreload 2

from scaling_llms.experiments import build_trainer
from scaling_llms.trainer import TrainerConfig
from scaling_llms.models import GPTConfig, GPTModel
from scaling_llms.utils.io import load_module_from_path, get_local_repo_dir
from scaling_llms.data import get_vocab_size
import gc
import torch
from torch.utils.data import IterableDataset, DataLoader


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
class RandomTokenDataset(IterableDataset):
    """
    Infinite stream of random token sequences.
    Useful for synthetic benchmarking / coord checks.
    """

    def __init__(self, vocab_size: int, seq_len: int, seed: int = 42):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.seed = seed

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed)

        while True:
            x = torch.randint(
                0,
                self.vocab_size,
                (self.seq_len,),
                generator=g,
            )
            y = torch.randint(
                0,
                self.vocab_size,
                (self.seq_len,),
                generator=g,
            )
            yield x, y


def make_synthetic_dataloader(
    batch_size: int,
    seq_len: int,
    vocab_size: int,
    seed: int = 42,
    num_workers: int = 0,
) -> DataLoader:
    """
    Returns an infinite DataLoader of random token batches.

    Output:
        x: (B, T)
        y: (B, T)
    """

    dataset = RandomTokenDataset(
        vocab_size=vocab_size,
        seq_len=seq_len,
        seed=seed,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
    )

In [ ]:
def fits_batch_size(
    batch_size: int,
    seq_len: int,
    vocab_size: int,
    width: int,
    gpt_hparams: dict,
    device: str = "cuda",
):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    trainer = None
    raw_model = None
    train_dl = None

    try:
        trainer_cfg = TrainerConfig(
            num_steps=1,
            accum_steps=1,
            precision="bf16",
            device=device,
            use_compile=False,
        )

        gpt_cfg = GPTConfig(
            n_embd=width,
            seq_len=seq_len,
            vocab_size=vocab_size,
            **gpt_hparams,
        )

        raw_model = GPTModel(gpt_cfg)

        train_dl = make_synthetic_dataloader(
            batch_size=batch_size,
            seq_len=seq_len,
            vocab_size=vocab_size,
        )

        trainer = build_trainer(
            cfg=trainer_cfg,
            raw_model=raw_model,
            train_dl=train_dl,
            eval_dl=None,
            run=None,
        )

        trainer.train(max_steps=1)

        peak_gb = torch.cuda.max_memory_allocated() / 1024**3
        return True, peak_gb

    except torch.cuda.OutOfMemoryError:
        return False, None

    finally:
        del trainer, raw_model, train_dl
        gc.collect()
        torch.cuda.empty_cache()


def find_max_batch_size(
    seq_len: int,
    vocab_size: int,
    width: int,
    gpt_hparams: dict,
    start_bs: int = 1,
    max_trials: int = 20,
    device: str = "cuda",
):
    
    CONSTANT_KWARGS = dict(
        seq_len=seq_len,
        vocab_size=vocab_size,
        width=width,
        gpt_hparams=gpt_hparams,
        device=device,
    )

    lo, hi = 0, start_bs

    # Exponential search upward
    for _ in range(max_trials):
        ok, peak_gb = fits_batch_size(
            batch_size=hi,
            **CONSTANT_KWARGS
        )
        print(f"Batch size {hi} -> {'Fits' if ok else 'OOM'}")
        if ok:
            lo = hi
            hi *= 2
        else:
            break

    # Binary search
    while lo + 1 < hi:
        mid = (lo + hi) // 2
        ok, peak_gb = fits_batch_size(
            batch_size=mid,
            **CONSTANT_KWARGS
        )
        print(f"Batch size {mid} -> {'Fits' if ok else 'OOM'}")
        if ok:
            lo = mid
        else:
            hi = mid

    ok, peak_gb = fits_batch_size(
        batch_size=lo,
        **CONSTANT_KWARGS
    )
    print(f"Batch size {lo} -> {'Fits' if ok else 'OOM'}")

    return lo, peak_gb

In [16]:
CONFIGS_ROOT = get_local_repo_dir() / "configs"
EXP_CONFIGS_ROOT = CONFIGS_ROOT / "experiments" /"mup_test"
RUNTIME_CONFIGS_ROOT = CONFIGS_ROOT / "runtime"
configs = load_module_from_path(EXP_CONFIGS_ROOT / "training_curves.py")

In [ ]:
max_bs, peak_gb = find_max_batch_size(
    seq_len=configs.DATALOADER_KWARGS["seq_len"],
    vocab_size=get_vocab_size(configs.DATASET_KWARGS["tokenizer_name"]),
    width=max(configs.WIDTHS),
    gpt_hparams=configs.CONSTANT_GPT_HPARAMS,
    start_bs=8,
)

print(max_bs, peak_gb)